# 9.3 Data in tables

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The table format of `data`, with indices running along the left and top edges and values
corresponding to pairs of indices, can be more concise or easier to read than the list format
described in the previous section. Here we describe tables first for two-dimensional
parameters and then for slices from higher-dimensional ones. We also show how the corresponding
multidimensional sets can be specified in tables that have entries of + or -
rather than parameter value entries.

AMPL also supports a convenient extension of the table format, in which more than
two indices may appear along the left and top edge. The rules for specifying such tables
are provided near the end of this section.

#### Two-dimensional tables

Data values for a parameter indexed over two sets, such as the shipping cost `data` from
the transportation `model` of [Figure 3-1a](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1a):

```ampl
set ORIG;
set DEST;
param cost {ORIG,DEST} >= 0;
```

are very naturally specified in a table ([Figure 3-1b](../03/3_2_an_AMPL_model_for_the_transportation_problem.ipynb#fig-3-1b)):

```ampl
param cost:  FRA   DET    LAN   WIN   STL   FRE   LAF :=
       GARY   39    14     11    14    16    82     8
       CLEV   27     9     12     9    26    95    17
       PITT   24    14     17    13    28    99    20 ;
```

The row labels give the first index and the column labels the second index, so that for
example `cost["GARY","FRA"]` is set to 39. To enable AMPL to recognize this as a
table, a colon must follow the parameter name, while the := operator follows the list of
column labels.

For larger index sets, the columns of tables become impossible to view within the
width of a single screen or page. To deal with this situation, AMPL offers several
alternatives, which we illustrate on the small table above.

When only one of the index sets is uncomfortably large, the table may be transposed
so that the column labels correspond to the smaller set:

```ampl
param cost (tr):
        GARY CLEV PITT :=
   FRA   39   27   24
   DET   14    9   14
   LAN   11   12   17
   WIN   14    9   13
   STL   16   26   28
   FRE   82   95   99
   LAF    8   17   20 ;
```

The notation (tr) after the parameter name indicates a transposed table, in which the
column labels give the first index and the row labels the second index. When both of the
index sets are large, either the table or its transpose may be divided up in some way.
Since line breaks are ignored, each row may be divided across several lines:

```ampl
param cost:    FRA   DET    LAN    WIN
               STL   FRE    LAF :=
         GARY   39    14     11     14
                16    82      8
         CLEV   27     9     12      9
                26    95     17
         PITT   24    14     17     13
                28    99     20           ;
```

Or the table may be divided columnwise into several smaller ones:

```ampl
param cost:   FRA   DET    LAN    WIN :=
       GARY    39    14     11     14
       CLEV    27     9     12      9
       PITT    24    14     17     13
            : STL   FRE    LAF :=
         GARY  16    82      8
         CLEV  26    95     17
         PITT  28    99     20 ;
```

A colon indicates the start of each new sub-table; in this example, each has the same row
labels, but a different subset of the column labels.

In the alternative formulation of this `model` presented in [Figure 6-2a](../06/6_2_subsets_and_slices_of_ordered_pairs.ipynb#fig-6-2a), cost is not
indexed over all combinations of members of ORIG and DEST, but over a subset of pairs
from these sets:

```ampl
set LINKS within {ORIG,DEST};
param cost {LINKS} >= 0;
```

As we have seen in [Section 9.2](./9_2_data_in_lists.ipynb), the membership of LINKS can be given concisely by a
list of pairs:

```ampl
set LINKS :=
   (GARY,*) DET LAN STL LAF
   (CLEV,*) FRA DET LAN WIN STL LAF
   (PITT,*) FRA WIN STL FRE ;
```

Rather than being given in a similar list, the values of cost can be given in a table like
this:

```ampl
param cost:  FRA   DET    LAN    WIN    STL    FRE    LAF :=
      GARY     .    14     11      .     16      .      8
      CLEV    27     9     12      9     26      .     17
      PITT    24     .      .     13     28     99      . ;
```

A cost value is given for all pairs that exist in LINKS, while a dot (.) serves as a
place-holder for pairs that are not in LINKS. The dot can appear in any AMPL table to
indicate "no value specified here".

The set LINKS may itself be given by a table that is analogous to the one for cost:

```ampl
set LINKS:   FRA   DET    LAN    WIN    STL    FRE    LAF :=
      GARY    -     +      +      -      +      -      +
      CLEV    +     +      +      +      +      -      +
      PITT    +     -      -      +      +      +      - ;
```

A + indicates a pair that is a member of the set, and a - indicates a pair that is not a member.
Any of AMPL's table formats for specifying parameters can be used for sets in this
way.

Two-dimensional slices of higher-dimensional `data`
To provide `data` for parameters of more than two dimensions, we can specify the values
in two-dimensional slices that are represented as tables. The rules for using slices are
much the same as for lists. As an example, consider again the three-dimensional parameter
cost defined by

```ampl
set ROUTES within {ORIG,DEST,PROD};
param cost {ROUTES} >= 0;
```

The values for this parameter that we specified in list format in the previous section as

```ampl
param cost :=
   [*,*,bands]   CLEV FRA 27  CLEV DET 9   CLEV LAN 12
                 CLEV STL 26  CLEV LAF 17  PITT FRA 24
                 PITT WIN 13  PITT STL 28  PITT FRE 99
    [*,*,coils]  GARY LAN 11  GARY STL 16  GARY LAF 8
                 CLEV FRA 23  CLEV DET 8   CLEV LAN 10
                 CLEV WIN 9   CLEV STL 21  PITT FRE 81
```

can instead be written in table format as

```ampl
param cost :=
 [*,*,bands]: FRA   DET    LAN    WIN    STL    FRE    LAF :=
        CLEV   27     9     12      .     26      .     17
        PITT   24     .      .     13     28     99      .
 [*,*,coils]: FRA   DET    LAN    WIN    STL    FRE    LAF :=
        GARY    .     .     11      .     16      .      8
        CLEV   23     8     10      9     21      .      .
        PITT    .     .      .      .      .     81      . ;
```

Since we are working with two-dimensional tables, there must be two \*'s in the templates.
A table value's row label is substituted for the first \*, and its column label for the
second, unless the opposite is specified by (tr) right after the template. You can omit
any rows or columns that would have no significant entries, such as the row for GARY in
the `[*,*,bands]` table above.

As before, a dot in the table for any slice indicates a tuple that is not a member of the
table.

An analogous table to specify the set ROUTES can be constructed by putting a +
where each number appears:

```ampl
set ROUTES :=
 (*,*,bands): FRA DET LAN WIN STL FRE LAF :=
        CLEV   +   +   +   -   +   -   +
        PITT   +   -   -   +   +   +   -
 (*,*,coils): FRA DET LAN WIN STL FRE LAF :=
        GARY   -   -   +   -   +   -   +
        CLEV   +   +   +   +   +   -   -
        PITT   -   -   -   -   -   +   - ;
```

Since the templates are now set templates rather than parameter templates, they are
enclosed in parentheses rather than brackets.

**Higher-dimensional tables**

By putting more than one index to the left of each row or at the top of each column,
you can describe multidimensional `data` in a single table rather than a series of slices.
We'll continue with the three-dimensional cost `data` to illustrate some of the wide variety
of possibilities.

By putting the first two indices, from sets ORIG and DEST, to the left, with the third
index from set PROD at the top, we produce the following three-dimensional table of the
costs:

```ampl
param cost: bands coils :=
  CLEV FRA    27    23
  CLEV DET     8     8
  CLEV LAN    12    10
  CLEV WIN     .     9
  CLEV STL    26    21
  CLEV LAF    17     .
  PITT FRA    24     .
  PITT WIN    13     .
  PITT STL    28     .
  PITT FRE    99    81
  GARY LAN     .    11
  GARY STL     .    16
  GARY LAF     .     8 ;
```

Putting only the first index to the left, and the second and third at the top, we arrive
instead at the following table, which for convenience we break into two pieces:

```ampl
param cost: FRA    DET   LAN   WIN   STL   FRE   LAF
          : bands bands bands bands bands bands bands :=
       CLEV   27     9    12     .    26     .    17
       PITT   24     .     .    13    28    99     .
             : FRA    DET   LAN   WIN   STL   FRE   LAF
             : coils coils coils coils coils coils coils :=
          GARY    .     .    11     .    16     .     8
          CLEV   23     8    10     9    21     .     .
          PITT    .     .     .     .     .    81     . ;
```

In general a colon must precede each of the table heading lines, while a := is placed only
after the last heading line.

The indices are taken in the order that they appear, first at the left and then at the top,
if no indication is given to the contrary. As with other tables, you can add the indicator
(tr) to transpose the table, so that the indices are still taken in order but first from the
top and then from the left:

```ampl
param cost (tr): CLEV CLEV CLEV CLEV CLEV CLEV
               : FRA DET LAN WIN STL LAF :=
          bands    27    8   12    .   26   17
          coils    23    8   10    9   21    .
                   : PITT PITT PITT PITT GARY GARY GARY
                   : FRA WIN STL FRE LAN STL LAF :=
              bands    24   13   28   99    .    .    .
              coils     .    .    .   81   11   16    8 ;
```

Templates can also be used to specify more precisely what goes where. For multidimensional
tables the template has two symbols in it, * to indicate those indices that appear at
the left and : to indicate those that appear at the top. For example the template
`[*,:,*]` gives a representation in which the first and third indices are at the left and the
second is at the top:

```ampl
param cost :=
   [*,:,*] : FRA   DET    LAN    WIN    STL    FRE    LAF :=
  CLEV bands  27     9     12      .     26      .     17
  CLEV coils  23     8     10      9     21      .      .
  PITT bands  24     .      .     13     28     99      .
  PITT coils   .     .      .      .      .     81      .
  GARY coils   .     .     11      .     16      .      8 ;
```

The ordering of the indices is always preserved in tables of this kind. The third index is
never correctly placed before the first, for example, no matter what transposition or templates
are employed.

For parameters of four or more dimensions, the ideas of slicing and multidimensional
tables can be applied together provide an especially broad choice of table formats. If
cost were indexed over ORIG, DEST, PROD, and 1..T, for instance, then the templates
`[*,:,bands,*]` and `[*,:,coils,*]` could be used to specify two slices through
the third index, each specified by a multidimensional table with two indices at the left and
one at the top.

Choice of format
The arrangement of slices to represent multidimensional `data` has no effect on how the
`data` values are used in the `model`, so you can choose the most convenient format. For the
cost parameter above, it may be appealing to slice along the third dimension, so that the
`data` values are organized into one shipping-cost table for each product. Alternatively,
placing all of the origin-product pairs at the left gives a particularly concise representation.
As another example, consider the revenue parameter from [Figure 6-3](../06/6_5_indexed_collections_of_sets.ipynb#fig-6-3):

```ampl
set PROD;                # products
set AREA {PROD};         # market areas for each product
param T > 0;             # number of weeks
param revenue {p in PROD, AREA[p], 1..T} >= 0;
```

Because the index set AREA[p] is potentially different for each product p, slices through
the first (PROD) dimension are most attractive. In the sample `data` from [Figure 6-4](../06/6_5_indexed_collections_of_sets.ipynb#fig-6-4), they
look like this:

```ampl
param T := 4 ;
set PROD := bands coils ;
set AREA[bands] := east north ;
set AREA[coils] := east west export ;
param revenue :=
  [bands,*,*]:   1      2     3     4 :=
      east     25.0   26.0  27.0  27.0
      north    26.5   27.5  28.0  28.5
  [coils,*,*]:   1      2     3     4 :=
      east      30     35    37    39
      west      29     32    33    35
      export    25     25    25    28 ;
```

We have a separate revenue table for each product p, with market areas from AREA[p]
labeling the rows, and weeks from 1..T labeling the columns.